# TrialNav — Exploratory Data Analysis

Run `data/fetch_trials.py` and `data/preprocess_trials.py` before executing this notebook.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load Data

In [ ]:
with open('../data/processed_trials.json') as f:
    trials = json.load(f)

df = pd.DataFrame(trials)
print(f'Loaded {len(df)} trials')
df.head(3)

## 2. Dataset Summary

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nEmpty eligibility_raw:', (df['eligibility_raw'] == '').sum())

## 3. Text Length Distribution

In [ ]:
df['eligibility_len'] = df['eligibility_raw'].str.len()
df['summary_len'] = df['summary'].str.len()
df['eligibility_tokens'] = df['eligibility_raw'].str.split().str.len()

fig, axes = plt.subplots(1, 2)
df['eligibility_tokens'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Eligibility Criteria — Token Count')
axes[0].set_xlabel('Tokens')

df['summary_len'].hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Trial Summary — Character Count')
axes[1].set_xlabel('Characters')
plt.tight_layout()
plt.show()

print(df[['eligibility_tokens', 'eligibility_len', 'summary_len']].describe().round(1))

## 4. Condition Distribution

In [ ]:
all_conditions = [c for clist in df['conditions'] for c in clist]
cond_counts = Counter(all_conditions).most_common(20)
cond_df = pd.DataFrame(cond_counts, columns=['condition', 'count'])

plt.figure(figsize=(14, 6))
sns.barplot(data=cond_df, x='count', y='condition', palette='Blues_r')
plt.title('Top 20 Conditions in Trial Dataset')
plt.tight_layout()
plt.show()

## 5. Word Cloud — Eligibility Criteria

In [ ]:
all_text = ' '.join(df['eligibility_raw'].dropna())
wc = WordCloud(width=1200, height=500, background_color='white', colormap='Blues', max_words=150)
wc.generate(all_text)

plt.figure(figsize=(16, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Terms in Eligibility Criteria')
plt.show()

## 6. Phase and Gender Distribution

In [ ]:
fig, axes = plt.subplots(1, 2)

phase_counts = df['phase'].explode().value_counts()
phase_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Trial Phase Distribution')
axes[0].set_xlabel('Phase')
axes[0].tick_params(axis='x', rotation=45)

gender_counts = df['gender'].value_counts()
gender_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Gender Eligibility')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 7. Embedding Visualization (UMAP)

In [ ]:
# Requires the FAISS index to exist — run retrieval/embed_trials.py first
import pickle
import umap
import faiss

index = faiss.read_index('../data/faiss_index/trials.index')
with open('../data/faiss_index/trials_metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

# Reconstruct embeddings from FAISS index
n = min(index.ntotal, 500)  # Limit for speed
embeddings = np.zeros((n, index.d), dtype=np.float32)
for i in range(n):
    index.reconstruct(i, embeddings[i])

# UMAP projection
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
embedding_2d = reducer.fit_transform(embeddings)

# Color by first condition
top_conditions = [t.get('conditions', ['Unknown'])[0] if t.get('conditions') else 'Unknown' for t in meta[:n]]
unique_conds = list(set(top_conditions))
color_idx = [unique_conds.index(c) for c in top_conditions]

plt.figure(figsize=(14, 8))
scatter = plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=color_idx, cmap='tab20', alpha=0.6, s=10)
plt.title(f'UMAP of {n} Trial Embeddings (colored by primary condition)')
plt.xlabel('UMAP-1')
plt.ylabel('UMAP-2')
plt.tight_layout()
plt.show()

## 8. Age Range Analysis

In [ ]:
def parse_age(age_str):
    if not age_str:
        return None
    try:
        return int(str(age_str).replace(' Years', '').replace(' Year', '').strip())
    except ValueError:
        return None

df['min_age_int'] = df['min_age'].apply(parse_age)
df['max_age_int'] = df['max_age'].apply(parse_age)

fig, axes = plt.subplots(1, 2)
df['min_age_int'].dropna().hist(bins=20, ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Minimum Age Requirements')
axes[0].set_xlabel('Age')

df['max_age_int'].dropna().hist(bins=20, ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Maximum Age Requirements')
axes[1].set_xlabel('Age')
plt.tight_layout()
plt.show()